# 02x — Fine-tuning Stage 3.5 (targeted oversampling + label correction)

Lanjutan dari Stage 3 (`best.ckpt`) — BUKAN training dari nol.
Dijalankan untuk tiap arsitektur (ConvNeXt, SwinV2, MaxViT, EfficientNet) secara terpisah.

**Sumber data keputusan**: `finetune_stage35_plan.csv` dari NB00 Bagian 2.
- `action=oversample` → bobot sampling dinaikkan 3x
- `action=relabel` → label dikoreksi sebelum training

**Konfigurasi**: LR kecil (1e-6 ~ 5e-6), 4 epoch, backbone tidak dibekukan.
Validasi OOF setelah selesai untuk pastikan tidak merusak kelas lain.


In [ ]:
!pip install -q timm==1.0.11


In [ ]:
import os
os.kill(os.getpid(), 9)  # restart kernel setelah pip install


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import timm, timm.data
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| timm:', timm.__version__)
if DEVICE.type != 'cuda':
    print('PERINGATAN: GPU tidak aktif — fine-tuning akan sangat lambat.')


## CONFIG — ubah ARCH_KEY sesuai arsitektur yang mau di-fine-tune


In [ ]:
# ── Pilih satu arsitektur per run ───────────────────────────────────────────
ARCH_KEY = 'convnextv2_base'  # ganti ke: swinv2_base_window12to24 | maxvit_base_tf | tf_efficientnetv2_m

ARCH_SPECS = {
    'convnextv2_base': dict(
        timm_tag='convnextv2_base.fcmae_ft_in22k_in1k_384',
        drop_path_rate=0.3, drive_folder='ConvNeXt V2', n_folds=5,
    ),
    'swinv2_base_window12to24': dict(
        timm_tag='swinv2_base_window12to24_192to384.ms_in22k_ft_in1k',
        drop_path_rate=0.3, drive_folder='Swin Transformer V2', n_folds=2,
    ),
    'maxvit_base_tf': dict(
        timm_tag='maxvit_base_tf_384.in21k_ft_in1k',
        drop_path_rate=0.3, drive_folder='MaxViT', n_folds=2,
    ),
    'tf_efficientnetv2_m': dict(
        timm_tag='tf_efficientnetv2_m.in21k_ft_in1k',
        drop_path_rate=0.2, drive_folder='EfficientNetV2-M', n_folds=5,
    ),
}
SPEC = ARCH_SPECS[ARCH_KEY]

# ── Path ────────────────────────────────────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/Big Data Challenge - Satria Data 2026')
PROCESSED_ROOT = DRIVE_ROOT / 'Preprocessing_Output' / 'processed'
MANIFEST_DIR   = DRIVE_ROOT / 'Preprocessing_Output' / 'manifests'
EVIDENCE_ROOT  = DRIVE_ROOT / 'Preprocessing_Output' / 'diagnostik_hard_cases'
CKPT_ROOT      = DRIVE_ROOT / 'Training_Output'
FT_CKPT_ROOT   = DRIVE_ROOT / 'Training_Output_Stage35'  # checkpoint hasil fine-tuning disimpan di sini
FT_CKPT_ROOT.mkdir(parents=True, exist_ok=True)

FOLD_CSV  = MANIFEST_DIR / 'fold_assignment.csv'
PLAN_CSV  = EVIDENCE_ROOT / 'finetune_stage35_plan.csv'
RELABEL_CSV = EVIDENCE_ROOT / 'all_label_corrections.csv'

for p in [PROCESSED_ROOT, FOLD_CSV, PLAN_CSV]:
    if not p.exists():
        raise FileNotFoundError(f'{p} tidak ditemukan')

# ── Hyperparameter Stage 3.5 ─────────────────────────────────────────────────
NUM_CLASSES = 3
FT_LR       = 2e-6   # learning rate sangat kecil
FT_EPOCHS   = 4      # sedikit epoch — cukup untuk adapt, tidak overfit
BATCH_SIZE  = 16
NUM_WORKERS = 2

print(f'Arsitektur : {ARCH_KEY}')
print(f'Drive folder: {SPEC["drive_folder"]}')
print(f'n_folds    : {SPEC["n_folds"]}')
print(f'LR         : {FT_LR}')
print(f'Epochs     : {FT_EPOCHS}')


## Load fold assignment + terapkan koreksi label


In [ ]:
fold_df = pd.read_csv(FOLD_CSV)
print(f'fold_assignment awal: {len(fold_df)} gambar')
print(fold_df['label'].value_counts().sort_index())

# Terapkan koreksi label dari NB00 Bagian 2
if RELABEL_CSV.exists():
    relabel_df = pd.read_csv(RELABEL_CSV)
    relabel_map = dict(zip(relabel_df['image_id'], relabel_df['expected_label']))
    n_corrected = fold_df['image_id'].isin(relabel_map).sum()
    fold_df['label'] = fold_df.apply(
        lambda r: relabel_map.get(r['image_id'], r['label']), axis=1
    )
    print(f'\nLabel correction diterapkan: {n_corrected} gambar dikoreksi')
    print(fold_df['label'].value_counts().sort_index())
else:
    print('all_label_corrections.csv tidak ditemukan — skip relabel')

label_by_id = fold_df.set_index('image_id')['label'].to_dict()


## Load fine-tuning plan — hitung oversampling weight per gambar


In [ ]:
plan_df = pd.read_csv(PLAN_CSV)
oversample_map = dict(zip(plan_df['image_id'], plan_df['oversample_weight']))
print(f'Plan: {len(plan_df)} gambar ({(plan_df["action"]=="oversample").sum()} oversample, '
      f'{(plan_df["action"]=="relabel").sum()} relabel)')

# Per group info
for grp, sub in plan_df.groupby('pattern_group'):
    print(f'  {grp}: {len(sub)} gambar, sim range [{sub["similarity"].min():.3f}, {sub["similarity"].max():.3f}]')


## Staging gambar ke disk lokal (hindari bottleneck Drive)


In [ ]:
LOCAL_ROOT = Path('/content/local_staging_ft')
LOCAL_PROC = LOCAL_ROOT / 'processed'
LOCAL_PROC.mkdir(parents=True, exist_ok=True)

def parallel_copy(pairs, desc, n=64, timeout=60):
    def _one(s, d):
        d = Path(d)
        if d.exists() and d.stat().st_size > 0:
            return
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(s, d)
    errs = []
    with ThreadPoolExecutor(max_workers=n) as ex:
        futs = {ex.submit(_one, s, d): (s, d) for s, d in pairs}
        for f in tqdm(list(futs), total=len(futs), desc=desc):
            try:
                f.result(timeout=timeout)
            except Exception as e:
                errs.append(str(e))
    return errs

pairs = [(PROCESSED_ROOT / f'{iid}.jpg', LOCAL_PROC / f'{iid}.jpg')
         for iid in fold_df['image_id']]
errs = parallel_copy(pairs, 'salin processed ke lokal')
if errs:
    print(f'PERINGATAN: {len(errs)} file gagal disalin')
print(f'Staging selesai: {len(pairs)} gambar')


## Dataset + transform + WeightedRandomSampler


In [ ]:
class WasteDataset(Dataset):
    def __init__(self, image_ids, label_map, root, transform):
        self.image_ids = np.array(image_ids, dtype=np.int64)
        self.label_map = label_map
        self.root = root
        self.transform = transform

    def __len__(self): return len(self.image_ids)

    def __getitem__(self, idx):
        iid = int(self.image_ids[idx])
        with Image.open(f'{self.root}/{iid}.jpg') as im:
            tensor = self.transform(im.convert('RGB'))
        return tensor, self.label_map[iid], iid


def build_train_transform(img_size):
    return T.Compose([
        T.RandomResizedCrop(img_size, scale=(0.7, 1.0), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        T.RandomGrayscale(p=0.05),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


def build_val_transform(img_size):
    return T.Compose([
        T.Resize(round(img_size / 0.95), interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


def make_sampler(train_ids, oversample_map, default_w=1.0):
    weights = np.array([oversample_map.get(int(i), default_w) for i in train_ids], dtype=np.float64)
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)


print('Dataset & transform helpers siap.')


## Training loop Stage 3.5 (semua fold)


In [ ]:
results_per_fold = []

for fold_k in range(SPEC['n_folds']):
    print(f'\n{'='*60}')
    print(f'Fold {fold_k} — {ARCH_KEY}')
    print(f'{'='*60}')

    # ── Load best.ckpt Stage 3 ───────────────────────────────────────────────
    ckpt_path = CKPT_ROOT / SPEC['drive_folder'] / f'fold{fold_k}' / 'best.ckpt'
    if not ckpt_path.exists():
        print(f'SKIP fold {fold_k}: {ckpt_path} tidak ada')
        continue

    best = torch.load(ckpt_path, map_location='cpu')
    img_size = best['img_size']

    # Resolve normalization stats
    dummy = timm.create_model(SPEC['timm_tag'], pretrained=False, num_classes=NUM_CLASSES)
    data_cfg = timm.data.resolve_model_data_config(dummy)
    mean, std = list(data_cfg['mean']), list(data_cfg['std'])
    del dummy

    # Bangun model dari checkpoint
    kwargs = dict(pretrained=False, num_classes=NUM_CLASSES,
                  drop_path_rate=SPEC['drop_path_rate'], img_size=img_size)
    try:
        model = timm.create_model(SPEC['timm_tag'], **kwargs)
    except TypeError:
        kwargs.pop('img_size')
        model = timm.create_model(SPEC['timm_tag'], **kwargs)
    model.load_state_dict(best['weights'])
    model = model.to(DEVICE)

    # ── Split fold ───────────────────────────────────────────────────────────
    train_ids = fold_df[fold_df['fold'] != fold_k]['image_id'].values
    val_ids   = fold_df[fold_df['fold'] == fold_k]['image_id'].values

    train_ds = WasteDataset(train_ids, label_by_id, LOCAL_PROC, build_train_transform(img_size))
    val_ds   = WasteDataset(val_ids,   label_by_id, LOCAL_PROC, build_val_transform(img_size))

    sampler     = make_sampler(train_ids, oversample_map)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)

    print(f'Train: {len(train_ids)} gambar | Val: {len(val_ids)} gambar | img_size: {img_size}')
    n_plan_in_train = sum(1 for i in train_ids if int(i) in oversample_map)
    print(f'Gambar dari plan (oversample/relabel) di train set: {n_plan_in_train}')

    # ── Optimizer & scheduler ─────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS, eta_min=FT_LR/10)

    best_val_f1 = float(best.get('val_f1', 0.0))
    best_state  = {k: v.clone() for k, v in model.state_dict().items()}
    print(f'Stage 3 best val_f1 (baseline): {best_val_f1:.5f}')

    # ── Training epochs ───────────────────────────────────────────────────────
    for epoch in range(1, FT_EPOCHS + 1):
        # Train
        model.train()
        train_loss, n_train = 0.0, 0
        for imgs, labels, _ in tqdm(train_loader, desc=f'Epoch {epoch}/{FT_EPOCHS} train', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(imgs)
            n_train += len(imgs)
        scheduler.step()

        # Validate
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels, _ in tqdm(val_loader, desc=f'Epoch {epoch}/{FT_EPOCHS} val', leave=False):
                preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        print(f'  Epoch {epoch}: loss={train_loss/n_train:.4f}  val_macro_f1={val_f1:.5f}  lr={scheduler.get_last_lr()[0]:.2e}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}
            print(f'  *** New best val_f1={val_f1:.5f} — checkpoint updated')

    # ── Simpan checkpoint Stage 3.5 ───────────────────────────────────────────
    ft_fold_dir = FT_CKPT_ROOT / SPEC['drive_folder'] / f'fold{fold_k}'
    ft_fold_dir.mkdir(parents=True, exist_ok=True)
    ft_ckpt = {
        'weights': best_state,
        'img_size': img_size,
        'val_f1': best_val_f1,
        'arch': ARCH_KEY,
        'timm_tag': SPEC['timm_tag'],
        'stage': '3.5',
        'ft_lr': FT_LR,
        'ft_epochs': FT_EPOCHS,
    }
    torch.save(ft_ckpt, ft_fold_dir / 'best.ckpt')
    print(f'Checkpoint Stage 3.5 disimpan: {ft_fold_dir}/best.ckpt  (val_f1={best_val_f1:.5f})')

    results_per_fold.append({'fold': fold_k, 'val_f1': best_val_f1})

    del model
    torch.cuda.empty_cache()

print('\n=== RINGKASAN STAGE 3.5 ===')
for r in results_per_fold:
    print(f"  fold {r['fold']}: best val_macro_f1 = {r['val_f1']:.5f}")
if results_per_fold:
    avg = np.mean([r['val_f1'] for r in results_per_fold])
    print(f'  RATA-RATA: {avg:.5f}')


## Catatan penggunaan checkpoint Stage 3.5

Checkpoint hasil fine-tuning tersimpan di `Training_Output_Stage35/{NamaFolderArsitektur}/fold{k}/best.ckpt`.

Untuk menggunakannya di NB03 (ensemble) dan NB04 (inferensi), ubah `CKPT_ROOT` menjadi
`Training_Output_Stage35` — atau buat versi hybrid: gunakan Stage 3.5 jika val_f1 lebih
tinggi dari Stage 3, pertahankan Stage 3 jika sebaliknya.

Cek delta val_f1 (Stage 3.5 vs Stage 3) sebelum memutuskan checkpoint mana yang dipakai.
